In [1]:
library(Seurat)
library(SeuratDisk)
library(DESeq2)
library(tidyverse)
library(reshape2)
library(edgeR)
library(glue)


载入需要的程辑包：SeuratObject

载入需要的程辑包：sp


载入程辑包：‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat

载入需要的程辑包：S4Vectors

Warning message:
“程辑包‘S4Vectors’是用R版本4.3.3 来建造的”
载入需要的程辑包：stats4

载入需要的程辑包：BiocGenerics

Warning message:
“程辑包‘BiocGenerics’是用R版本4.3.2 来建造的”

载入程辑包：‘BiocGenerics’


The following object is masked from ‘package:SeuratObject’:

    intersect


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, uni

In [2]:
setwd('/media/AnalysisTempDisk2/Wuyisha/HNM_IMMUNE/RNA/DEG_My/DEseq2')
dir.create("DEG_MAR_vs_SAR_DEseq2/Raw")
dir.create("DEG_MAR_vs_SAR_DEseq2/Filter")
dir.create("DEG_MAR_vs_SAR_DEseq2/Fig")


Warning message in dir.create("DEG_MAR_vs_SAR_DEseq2/Raw"):
“'DEG_MAR_vs_SAR_DEseq2/Raw'已存在”
Warning message in dir.create("DEG_MAR_vs_SAR_DEseq2/Filter"):
“'DEG_MAR_vs_SAR_DEseq2/Filter'已存在”
Warning message in dir.create("DEG_MAR_vs_SAR_DEseq2/Fig"):
“'DEG_MAR_vs_SAR_DEseq2/Fig'已存在”


In [34]:
T_CD4 <- readRDS('/media/AnalysisTempDisk2/Wuyisha/HNM_IMMUNE/RNA/data/data_Myeloid.rds')
print(T_CD4)

An object of class Seurat 
21157 features across 15062 samples within 1 assay 
Active assay: RNA (21157 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 3 dimensional reductions calculated: pca, umap, harmony


In [4]:
Cells.sub <- subset(T_CD4@meta.data, group_A!="PAR")
T_CD4 <- subset(T_CD4, cells=row.names(Cells.sub))

In [35]:
Cells.sub <- subset(T_CD4@meta.data, celltype=="Macro_CCL3")
T_CD4 <- subset(T_CD4, cells=row.names(Cells.sub))

In [4]:
Cells.sub <- subset(T_CD4@meta.data, group_A!="PAR")
T_CD4 <- subset(T_CD4, cells=row.names(Cells.sub))

Cells.sub <- subset(T_CD4@meta.data, group_A!="MAR")
T_CD4 <- subset(T_CD4, cells=row.names(Cells.sub))
T_CD4

An object of class Seurat 
21157 features across 9978 samples within 1 assay 
Active assay: RNA (21157 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 3 dimensional reductions calculated: pca, umap, harmony

In [14]:
colnames(T_CD4@meta.data)

[1] "orig.ident"             "nCount_RNA"             "nFeature_RNA"          
 [4] "percent.mt"             "Doublet"                "sample"                
 [7] "RNA_snn_res.0.1"        "seurat_clusters"        "RNA_snn_res.0.2"       
[10] "RNA_snn_res.0.3"        "RNA_snn_res.0.5"        "RNA_snn_res.0.8"       
[13] "RNA_snn_res.1"          "celltype"               "barcodes"              
[16] "sample_bar"             "Group"                  "Age"                   
[19] "Sex"                    "sneezing"               "Runny_nose"            
[22] "Itchy_nose"             "Nasal_congestion"       "Eye_irritation"        
[25] "VAS_score"              "allergic_history"       "Total_IgE"             
[28] "Artemisia.annua.pollen" "Poplar.pollen"          "Elm.pollen"            
[31] "Willow.pollen"          "Dog.fur"                "Cat.fur"               
[34] "def"                    "dep"                    "group_a"               
[37] "group_A"                "group"                  "EOS_Count"

In [36]:
table(T_CD4$celltype)
table(T_CD4$group_A)
table(T_CD4$sample)

cell_types <- unique(T_CD4$celltype)
Idents(T_CD4) <- T_CD4$group_A


       Mast_cells        Macro_C1QB        Macro_CCL3        Macro_CPVL 
                0                 0              1779                 0 
       macro_SLPI       Macro_STAB1 Macro_prolifering           CD14_Mo 
                0                 0                 0                 0 
              cDC               pDC             migDC 
                0                 0                 0 


MAR NAR PAR SAR 
544 410  45 780 


AR04 AR05 AR07 AR09 AR10 AR14 AR17 AR19 AR20 AR21 AR22 AR23 AR24 AR25 AR26 AR27 
 132   93   75   71   51   13   63   63  107   59   28   30  646   81   73   18 
AR30 AR31 AR32 AR33 AR34 AR35 AR36 
  14   37   45   23   15   29   13 

In [37]:
## 按细胞类型拆分数据框，生成一个列表，每个元素对应一个细胞类型的数据
celltype_to_use <- unique(T_CD4$celltype)

# Assign sample names to object
Idents(T_CD4) <- paste(T_CD4$group_A, T_CD4$celltype,  T_CD4$celltype,sep = "_")
cts <- AggregateExpression(T_CD4, 
                           group.by = c("celltype", "sample",'group_A'),
                           assays = 'RNA',
                           slot = "counts",
                           return.seurat = FALSE)
cts <- cts$RNA

# Split into a list of PseudoBulk Matrix per cell type
cts.t <- t(cts)  # transpose
cts.t <- as.data.frame(cts.t)  # convert to data.frame


In [38]:
cts.t[0:6]

,ACKR3,HINT1,RRP36,NFKBIZ,EEF1A1,POSTN
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Macro-CCL3_AR04_MAR,2,124,17,275,3925,7
Macro-CCL3_AR05_MAR,1,57,8,149,3053,4
Macro-CCL3_AR07_MAR,5,88,11,251,4067,28
Macro-CCL3_AR09_NAR,0,50,3,138,1674,1
Macro-CCL3_AR10_NAR,0,64,5,212,2813,4
Macro-CCL3_AR14_MAR,9,15,2,56,968,0
Macro-CCL3_AR17_NAR,4,125,14,81,4655,11
Macro-CCL3_AR19_SAR,8,155,22,77,4266,16
Macro-CCL3_AR20_NAR,7,151,28,569,2805,6


In [39]:
# 1. 最简洁：按名字取列
vec <- cts.t[ , "HLA-DQA2"]        # 返回一个数值向量

# 2. 如果怕名字里有多余空格，先清洗再取
vec <- cts.t[ , colnames(cts.t) == "HLA-DQA2"]

# 3. 想保留矩阵的维度（1 列矩阵）：
sub_mat <- cts.t[ , "HLA-DQA2", drop = FALSE]   # 仍然是矩阵，列名保留

In [40]:
# 假设你的 data.frame 叫 sub_mat
rownames(sub_mat) <- sub("^[^_]*_([^_]+)_.*$", "\\1", rownames(sub_mat))

In [41]:
sub_mat

,HLA-DQA2
,<dbl>
AR04,46
AR05,123
AR07,375
AR09,14
AR10,6
AR14,24
AR17,27
AR19,3
AR20,85


In [57]:
meta <- read.csv('/media/AnalysisTempDisk2/Wuyisha/HNM_IMMUNE/RNA/data/meta.data.csv',row.names = 1)
head(meta)

,Group,Age,Sex,sneezing,Runny_nose,Itchy_nose,Nasal_congestion,Eye_irritation,VAS_score,allergic_history,⋯,Willow.pollen,Dog.fur,Cat.fur,def,dep,allergen,group_a,group_A,group,EOS_Count
,<chr>,<int>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<dbl>
AR04,AR,33,Female,1,2,2,10,1,16,0,⋯,0,NA,4,0,0,7,PAR,MAR,PER,0.09
AR05,AR,35,Male,5,6,6,5,4,26,0,⋯,3,0,2,2,2,15,PAR,MAR,PER,0.19
AR07,AR,29,Male,8,3,4,7,4,26,0,⋯,2,0,3,3,3,14,PAR,MAR,PER,0.22
AR09,NAR,60,Female,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,NAR,NAR,NAR,0.04
AR10,NAR,14,Male,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,NAR,NAR,NAR,0.04
AR14,AR,35,Female,2,10,3,9,0,24,0,⋯,1,0,3,3,0,10,PAR,MAR,PER,0.09


In [58]:
# 保证 sub_mat 是 data.frame（若是 matrix 先转）
if (is.matrix(sub_mat)) sub_mat <- as.data.frame(sub_mat)

# 按行名合并，保留 meta 的行顺序
res <- cbind(meta, sub_mat[rownames(meta), , drop = FALSE])

head(res)          # 查看结果

,Group,Age,Sex,sneezing,Runny_nose,Itchy_nose,Nasal_congestion,Eye_irritation,VAS_score,allergic_history,⋯,Dog.fur,Cat.fur,def,dep,allergen,group_a,group_A,group,EOS_Count,HLA-DQA2
,<chr>,<int>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>
AR04,AR,33,Female,1,2,2,10,1,16,0,⋯,NA,4,0,0,7,PAR,MAR,PER,0.09,46
AR05,AR,35,Male,5,6,6,5,4,26,0,⋯,0,2,2,2,15,PAR,MAR,PER,0.19,123
AR07,AR,29,Male,8,3,4,7,4,26,0,⋯,0,3,3,3,14,PAR,MAR,PER,0.22,375
AR09,NAR,60,Female,0,0,0,0,0,0,0,⋯,0,0,0,0,0,NAR,NAR,NAR,0.04,14
AR10,NAR,14,Male,0,0,0,0,0,0,0,⋯,0,0,0,0,0,NAR,NAR,NAR,0.04,6
AR14,AR,35,Female,2,10,3,9,0,24,0,⋯,0,3,3,0,10,PAR,MAR,PER,0.09,24


In [59]:
res

,Group,Age,Sex,sneezing,Runny_nose,Itchy_nose,Nasal_congestion,Eye_irritation,VAS_score,allergic_history,⋯,Dog.fur,Cat.fur,def,dep,allergen,group_a,group_A,group,EOS_Count,HLA-DQA2
,<chr>,<int>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>
AR04,AR,33,Female,1,2,2,10,1,16,0,⋯,NA,4,0,0,7,PAR,MAR,PER,0.09,46
AR05,AR,35,Male,5,6,6,5,4,26,0,⋯,0,2,2,2,15,PAR,MAR,PER,0.19,123
AR07,AR,29,Male,8,3,4,7,4,26,0,⋯,0,3,3,3,14,PAR,MAR,PER,0.22,375
AR09,NAR,60,Female,0,0,0,0,0,0,0,⋯,0,0,0,0,0,NAR,NAR,NAR,0.04,14
AR10,NAR,14,Male,0,0,0,0,0,0,0,⋯,0,0,0,0,0,NAR,NAR,NAR,0.04,6
AR14,AR,35,Female,2,10,3,9,0,24,0,⋯,0,3,3,0,10,PAR,MAR,PER,0.09,24
AR17,NAR,19,Male,0,0,1,3,0,4,0,⋯,0,0,0,0,0,NAR,NAR,NAR,0.12,27
AR19,AR,39,Male,9,8,9,10,5,41,0,⋯,0,0,0,0,2,SAR,SAR,PER,0.27,3
AR20,NAR,26,Male,0,1,0,0,0,1,0,⋯,0,0,0,0,0,NAR,NAR,NAR,0.04,85


In [87]:
res$DQA2 <- log2(res$`HLA-DQA2`)

In [88]:
colnames(res)

[1] "Group"                  "Age"                    "Sex"                   
 [4] "sneezing"               "Runny_nose"             "Itchy_nose"            
 [7] "Nasal_congestion"       "Eye_irritation"         "VAS_score"             
[10] "allergic_history"       "Total_IgE"              "Artemisia.annua.pollen"
[13] "Poplar.pollen"          "Elm.pollen"             "Willow.pollen"         
[16] "Dog.fur"                "Cat.fur"                "def"                   
[19] "dep"                    "allergen"               "group_a"               
[22] "group_A"                "group"                  "EOS_Count"             
[25] "HLA-DQA2"               "DQA2"

In [78]:
setwd('/media/AnalysisTempDisk2/Wuyisha/HNM_IMMUNE/RNA/DEG_My/HLA')

In [96]:
library(ggplot2)
library(ggpubr)   # 为了 stat_cor

symptoms <- c('VAS_score','Total_IgE', 'EOS_Count','allergen', "sneezing", "Runny_nose", "Itchy_nose",
              "Nasal_congestion", "Eye_irritation")

for (symp in symptoms) {
  # 1. 临时数据框
  plot_df <- data.frame(
    VAS = res[[symp]],            # 动态 y
    DQA = res$DQA2,
    grp = factor(res$group_A)
  )
  
  # 2. 画图
  p <- ggplot(plot_df, aes(x = DQA, y = VAS)) +
    geom_point(aes(shape = grp, fill = grp),
               colour = "#6A44AB", size = 3, stroke = 1.2, alpha = 0.8) +
    geom_smooth(method = "lm", se = TRUE, colour = "firebrick") +
    stat_cor(method = "spearman") +
    labs(title = paste("Correlation between HLA-DQA2 expression and", symp),
         x = "HLA-DQA2 expression",
         y = symp) +
    theme_minimal(base_size = 14) +
    theme(panel.grid  = element_blank(),
          axis.line.x = element_line(colour = "black", size = 1),
          axis.line.y = element_line(colour = "black", size = 1),
          panel.border = element_rect(colour = "black", fill = NA, size = 1),
          legend.title = element_blank())
  
  # 3. 保存
  ggsave(filename = paste0("correlation_HLA-DQA2_and_", symp, ".pdf"),
         plot = p, width = 5, height = 4.5)
}

`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'


In [ ]:
res_MAR <- res[res$group_A == "MAR", ]

In [7]:
# 按 "_" 拆开行名，取倒数第2 个元素
celltype_tags <- vapply(strsplit(rownames(cts.t), "_"),
                        function(x) x[length(x)-2],
                        character(1))
cts.split <- split(cts.t, celltype_tags)

In [8]:
counts_cellType <- lapply(cts.split, function(x){
    rownames(x) <- gsub(".*_AR", "AR", rownames(x))  # 提取AR之后的内容
    t(x)
})

In [9]:
T_CD4$samples_group <- paste(T_CD4$sample, T_CD4$group_A,sep = "_")

samples_table <- table(T_CD4@meta.data$celltype, T_CD4@meta.data$samples_group)
samples_df <- as.data.frame(samples_table)
samples_df <- dcast(samples_df, Var1 ~ Var2, value.var = "Freq")
rownames(samples_df) <- samples_df$Var1
samples_df$Var1 <- NULL

# samples_df 已经是“行=细胞类型，列=样本”的矩阵/数据框
keep_ct <- rownames(samples_df)[rowSums(samples_df < 10) == 0]   # 所有样本≥10
samples_df.filt <- samples_df[keep_ct, ]

In [10]:
samples_df.filt

,AR09_NAR,AR10_NAR,AR17_NAR,AR19_SAR,AR20_NAR,AR22_NAR,AR23_NAR,AR24_SAR,AR30_SAR,AR31_NAR,AR33_NAR,AR34_SAR,AR35_SAR,AR36_SAR
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
Macro_C1QB,51,23,112,67,54,34,40,331,13,31,44,22,27,35
Macro_CCL3,71,51,63,63,107,28,30,646,14,37,23,15,29,13
CD14_Mo,127,61,62,68,88,34,121,1211,20,19,136,71,109,89
cDC,151,75,68,36,150,32,50,407,63,79,25,29,31,31


In [11]:
cell_type_remain <- rownames(samples_df.filt)
cell_type_remain

[1] "Macro_C1QB" "Macro_CCL3" "CD14_Mo"    "cDC"

In [12]:
counts_cellType <- as.data.frame(counts_cellType)

In [39]:
#筛选出在超过 1% 的样本中 CPM 值小于 1 的基因
#筛选出总读数大于或等于 20 的基因
dge <- DGEList(counts = counts_cellType)
cpm_matrix <- cpm(dge)
cpm_percent <- rowSums(cpm_matrix<1) / ncol(cpm_matrix)  # % of samples that have <0.5 CPM value for each gene
remove_genes <- names(cpm_percent[cpm_percent > 1])
counts_cellType <- counts_cellType[!(rownames(counts_cellType) %in% remove_genes), ]

# filter
keep <- rowSums(counts_cellType) >= 30  ## PTSD: keeping only genes that have 20 reads and more
counts_cellType_filtered <- counts_cellType[keep,]

In [40]:
meta <- T_CD4@meta.data
cov_meta <- unique(meta[, c("sample", "Age", "Sex")])   # 保留需要的列
colData  <- merge(colData, cov_meta, by = "sample", all.x = TRUE)

In [41]:
library(stringr)
library(SummarizedExperiment)   # 仅用到 match 等基础函数，可省
meta <- T_CD4@meta.data
## 1. 从列名一次性解析出 3 个信息
raw_nm   <- colnames(counts_cellType_filtered)   # 原始列名
parts    <- str_split(raw_nm, "\\.", simplify = TRUE)

colData  <- data.frame(
    samples   = raw_nm,                    # 原始名字先留着
    celltype  = parts[, 1],                # 第一段
    sample    = parts[, ncol(parts)-1],    # 倒数第二段
    group_A   = parts[, ncol(parts)]       # 最后一段
)

## 2. 把行名做成“仅最后一个下划线”版本
rownames(colData) <- str_replace(raw_nm, "_(?=[^_]*$)", "\x01") %>% 
                     str_replace_all("_", ".") %>% 
                     str_replace_all("\x01", "_")

## 3. 直接按列顺序对齐（保证 1:1 对应）
colData <- colData[match(colnames(counts_cellType_filtered), rownames(colData)), ]
colData$sample <- str_extract(colData$sample, "[^.]*$")
## 4. 合并年龄、性别等临床信息（关键步骤）
## 假设 meta 里已有 sample/Age/Sex 列
cov_meta <- unique(meta[, c("sample", "Age", "Sex")])   # 保留需要的列
colData  <- merge(colData, cov_meta, by = "sample", all.x = TRUE)

## 5. 可选：快速查看每列唯一值
sapply(colData, function(x) length(unique(x)))

sample  samples celltype  group_A      Age      Sex 
      22      142        7       15        1        1

In [42]:
library(stringr)

raw_nm <- colnames(counts_cellType_filtered)

## 1. 提取三段
colData <- data.frame(
    samples = raw_nm,
    stringsAsFactors = FALSE
)

## 正则：非贪婪匹配到最后一串小数点前的内容 = celltype
colData$celltype  <- str_extract(raw_nm, "^(.*?)(?=\\.[^.]+_[^_]+$)")
rest              <- str_remove(raw_nm, "^(.*?)\\.")          # 去掉 celltype 部分

## 剩余字符串再拆两段
colData$sample    <- str_extract(rest, "^.*(?=_[^_]*$)")     # 最后一个下划线之前
colData$group_A   <- str_extract(rest, "[^_]+$")              # 最后一个下划线之后

## 2. 行名：仅保留最后一个下划线，其余 _ → .
rownames(colData) <- str_replace(raw_nm, "_(?=[^_]*$)", "\x01") %>% 
                     str_replace_all("_", ".") %>% 
                     str_replace_all("\x01", "_")

## 3. 列顺序对齐
colData <- colData[match(colnames(counts_cellType_filtered), rownames(colData)), ]
colData$sample <- str_extract(colData$sample, "[^.]*$")
## 4. 快速检查
sapply(colData, function(x) length(unique(x)))

samples celltype   sample  group_A 
     142       11       14        2

In [43]:
colData

,samples,celltype,sample,group_A
,<chr>,<chr>,<chr>,<chr>
CD14.Mo.AR09_NAR,CD14.Mo.AR09_NAR,CD14.Mo,AR09,NAR
CD14.Mo.AR10_NAR,CD14.Mo.AR10_NAR,CD14.Mo,AR10,NAR
CD14.Mo.AR17_NAR,CD14.Mo.AR17_NAR,CD14.Mo,AR17,NAR
CD14.Mo.AR19_SAR,CD14.Mo.AR19_SAR,CD14.Mo,AR19,SAR
CD14.Mo.AR20_NAR,CD14.Mo.AR20_NAR,CD14.Mo,AR20,NAR
CD14.Mo.AR22_NAR,CD14.Mo.AR22_NAR,CD14.Mo,AR22,NAR
CD14.Mo.AR23_NAR,CD14.Mo.AR23_NAR,CD14.Mo,AR23,NAR
CD14.Mo.AR24_SAR,CD14.Mo.AR24_SAR,CD14.Mo,AR24,SAR
CD14.Mo.AR30_SAR,CD14.Mo.AR30_SAR,CD14.Mo,AR30,SAR


In [44]:
meta <- T_CD4@meta.data
cov_meta <- unique(meta[, c("sample", "Age", "Sex")])   # 保留需要的列
colData  <- merge(colData, cov_meta, by = "sample", all.x = TRUE)

In [45]:
colData

sample,samples,celltype,group_A,Age,Sex
<chr>,<chr>,<chr>,<chr>,<int>,<chr>
AR09,CD14.Mo.AR09_NAR,CD14.Mo,NAR,60,Female
AR09,cDC.AR09_NAR,cDC,NAR,60,Female
AR09,Macro.CCL3.AR09_NAR,Macro.CCL3,NAR,60,Female
AR09,Macro.CPVL.AR09_NAR,Macro.CPVL,NAR,60,Female
AR09,pDC.AR09_NAR,pDC,NAR,60,Female
AR09,migDC.AR09_NAR,migDC,NAR,60,Female
AR09,Macro.prolifering.AR09_NAR,Macro.prolifering,NAR,60,Female
AR09,Macro.STAB1.AR09_NAR,Macro.STAB1,NAR,60,Female
AR09,Mast.cells.AR09_NAR,Mast.cells,NAR,60,Female


In [46]:
cell_type_remain <- str_replace_all(cell_type_remain, "\\_", ".")

In [47]:
cell_type_remain

[1] "Macro.C1QB" "Macro.CCL3" "CD14.Mo"    "cDC"

In [51]:
# 用于存储每个 celltype_L2 的分析结果
results_list <- list()

# 遍历每个 celltype_L2
for (celltype in cell_type_remain) {
    # 提取当前 celltype_L2 对应的样本
    current_samples <- colData$samples[colData$celltype == celltype]
    
    # 提取对应的 count 数据
    current_counts <- counts_cellType_filtered[, current_samples]
    
    # 提取对应的 colData
    current_colData <- colData[colData$celltype == celltype, ]
    
    # 创建 DESeqDataSet 对象
    dds <- DESeqDataSetFromMatrix(countData = current_counts,
                                  colData = current_colData,
                                  design = ~Age + Sex + group_A)
    
    dds$group_A <- relevel(as.factor(dds$group_A), 'NAR')
    
    # 运行 DESeq2
    dds <- DESeq(dds, 
                 test = "LRT",
                 reduced = ~Age + Sex)
    
    # 获取结果
    res <- results(dds)
    
    # 将结果存储到列表中
    results_list[[celltype]] <- res
}

# 查看每个 celltype_L2 的分析结果
for (celltype in names(results_list)) {
    cat("\nResults for", celltype, ":\n")
    print(head(results_list[[celltype]]))
}

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”


ERROR: Error in DESeq(dds, test = "LRT"): likelihood ratio test requires a 'reduced' design, see ?DESeq


In [52]:
?DESeq

DESeq {DESeq2},R Documentation
object,"a DESeqDataSet object, see the constructor functions DESeqDataSet, DESeqDataSetFromMatrix, DESeqDataSetFromHTSeqCount."
test,"either ""Wald"" or ""LRT"", which will then use either Wald significance tests (defined by nbinomWaldTest), or the likelihood ratio test on the difference in deviance between a full and reduced model formula (defined by nbinomLRT)"
fitType,"either ""parametric"", ""local"", ""mean"", or ""glmGamPoi"" for the type of fitting of dispersions to the mean intensity. See estimateDispersions for description."
sfType,"either ""ratio"", ""poscounts"", or ""iterate"" for the type of size factor estimation. See estimateSizeFactors for description."
betaPrior,"whether or not to put a zero-mean normal prior on the non-intercept coefficients See nbinomWaldTest for description of the calculation of the beta prior. In versions >=1.16, the default is set to FALSE, and shrunken LFCs are obtained afterwards using lfcShrink."
full,"for test=""LRT"", the full model formula, which is restricted to the formula in design(object). alternatively, it can be a model matrix constructed by the user. advanced use: specifying a model matrix for full and test=""Wald"" is possible if betaPrior=FALSE"
reduced,"for test=""LRT"", a reduced formula to compare against, i.e., the full formula with the term(s) of interest removed. alternatively, it can be a model matrix constructed by the user"
quiet,whether to print messages at each step
minReplicatesForReplace,"the minimum number of replicates required in order to use replaceOutliers on a sample. If there are samples with so many replicates, the model will be refit after these replacing outliers, flagged by Cook's distance. Set to Inf in order to never replace outliers. It set to Inf for fitType=""glmGamPoi""."
modelMatrixType,"either ""standard"" or ""expanded"", which describe how the model matrix, X of the GLM formula is formed. ""standard"" is as created by model.matrix using the design formula. ""expanded"" includes an indicator variable for each level of factors in addition to an intercept. for more information see the Description of nbinomWaldTest. betaPrior must be set to TRUE in order for expanded model matrices to be fit."


In [49]:
CD4_Trm  <- results_list$Macro.CCL3
CD4_Trm

log2 fold change (MLE): group A SAR vs NAR 
LRT p-value: '~ Age + Sex + group_A' vs '~ Age + Sex' 
DataFrame with 14904 rows and 6 columns
         baseMean log2FoldChange     lfcSE      stat    pvalue      padj
        <numeric>      <numeric> <numeric> <numeric> <numeric> <numeric>
ACKR3     2.06418       0.461699  1.013588  0.221336 0.6380240         1
HINT1    59.51100       0.207214  0.319529  0.452197 0.5012936         1
RRP36     4.62257       0.561605  0.641410  0.878381 0.3486456         1
NFKBIZ  118.71055      -0.699427  0.404238  2.770481 0.0960173         1
EEF1A1 2172.11763       0.175232  0.338761  0.283109 0.5946704         1
...           ...            ...       ...       ...       ...       ...
IFNA16   0.000000             NA        NA        NA        NA        NA
IFNA17   0.000000             NA        NA        NA        NA        NA
IFNA13   0.127398      -0.443881   3.33963 0.0852601  0.770292         1
AVP      0.000000             NA        NA        NA      

In [50]:
sorted_res_sig <- CD4_Trm[(!is.na(CD4_Trm$padj)) & (CD4_Trm$padj < 0.1), ]
sorted_res_sig

log2 fold change (MLE): group A SAR vs NAR 
LRT p-value: '~ Age + Sex + group_A' vs '~ Age + Sex' 
DataFrame with 2 rows and 6 columns
       baseMean log2FoldChange     lfcSE      stat      pvalue        padj
      <numeric>      <numeric> <numeric> <numeric>   <numeric>   <numeric>
IGHG1   33.0895       2.952503  0.678492   19.2638 1.13846e-05 8.45189e-02
IFIT1   14.3984      -0.109277  1.555806   43.5021 4.23500e-11 6.28812e-07

In [ ]:
##交集基因展示

In [88]:
marker_tb<- read.csv('/media/AnalysisTempDisk2/Wuyisha/HNM_IMMUNE/RNA/DEG_My/DEG_PAR_vs_NAR/Filter/Filter_marker_genes_Macro_CCL3_PAR_vs_NAR.csv')
head(marker_tb)

,X,p_val,avg_log2FC,pct.1,pct.2,p_val_adj
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,HLA-DQA2,2.771879e-45,1.776696,0.656,0.278,5.864465e-41
2,MT2A,1.565762e-41,-3.044271,0.338,0.693,3.312683e-37
3,S100P,1.201623e-27,-1.068151,0.221,0.522,2.542274e-23
4,LCN2,1.686062e-26,-1.558028,0.107,0.376,3.567201e-22
5,CFB,9.288274e-25,-1.098420,0.074,0.322,1.965120e-20
6,IFI6,1.092659e-24,-1.487515,0.127,0.395,2.311740e-20


In [94]:
DEseq2_up <- rownames(sorted_res_sig)
 DEseq2_up

[1] "MAT2A"    "HLA-DQA2"

In [98]:
marker_tb<- read.csv('/media/AnalysisTempDisk2/Wuyisha/HNM_IMMUNE/RNA/DEG_My/DEG_PAR_vs_NAR/Filter/Filter_marker_genes_Macro_CCL3_PAR_vs_NAR.csv')

Wilcoxon_up <- marker_tb[marker_tb$avg_log2FC > 0.25 & marker_tb$p_val_adj < 0.05, ]

Wilcoxon_up <- Wilcoxon_up$X
DEseq2_up <- rownames(sorted_res_sig)


####up_motif####
## 找到交集
common_names <- intersect(Wilcoxon_up,DEseq2_up)

unique_Wilcoxon <- setdiff(Wilcoxon_up, DEseq2_up)

unique_DEseq2 <- setdiff(DEseq2_up, Wilcoxon_up)

# 计算每个类别的数量
common_count <- length(common_names)
unique_W_count <- length(unique_Wilcoxon)
unique_DEseq_count <- length(unique_DEseq2)

# 将结果合并到一个数据框中
results <- data.frame(
  Type = c(rep("Common", common_count),
           rep("Unique to Wilcoxon", unique_W_count),
           rep("Unique to DEseq2", unique_DEseq_count)),
  Names = c(common_names, unique_W_count, unique_DEseq_count)
)

# 打印结果
print("Common names:")
print(common_names)
print("Unique names in Wilcoxon:")
print(unique_W_count)
print("Unique names in DEseq2:")
print(unique_DEseq_count)


## 韦恩图绘制
# 创建一个列表来存储这些子集
venn_data <- list(Wilcoxon_up,DEseq2_up)

#install.packages("VennDiagram")
library(VennDiagram)
# 绘制韦恩图
# 计算每个集合的百分比
total_elements <- length(unique(unlist(venn_data)))
percentages <- sapply(venn_data, function(x) length(x) / total_elements * 100)


# 创建Venn图
venn.plot <- venn.diagram(
  x = venn_data,
  category.names = c("Wilcoxon", "DEseq2"),
  filename = NULL,
  output = TRUE,
  height = 8, width = 10, resolution = 300,
  fill = c("#51A3A3", "#D5CA67"), # 设置每个集合的颜色
  cat.cex = 1.5, # 设置类别标签的字体大小
  cex = 1.5, # 设置数字的字体大小
  cat.pos = 0, # 设置类别标签的位置
  cat.dist = 0.05, # 设置类别标签的距离
  rotation.degree = 0, # 设置旋转角度
  label.col = "black", # 设置标签颜色
  cat.col = c( "green", "yellow"), # 设置类别标签颜色
  print.mode = c("raw", "percent"), # 显示百分比
  sigdigs = 2 # 设置百分比的小数位数
)

# 保存为PDF
pdf("./venn_diagram_up_MAR vs NAR getmarkerfeature.pdf")
grid.draw(venn.plot)
dev.off()



[1] "Common names:"
[1] "HLA-DQA2" "MAT2A"   
[1] "Unique names in Wilcoxon:"
[1] 46
[1] "Unique names in DEseq2:"
[1] 0


pdf 
  2

In [ ]:
##mar vs NAR

In [66]:
Cells.sub <- subset(T_CD4@meta.data, group_A!="PAR")
T_CD4 <- subset(T_CD4, cells=row.names(Cells.sub))

Cells.sub <- subset(T_CD4@meta.data, group_A!="SAR")
T_CD4 <- subset(T_CD4, cells=row.names(Cells.sub))
T_CD4

An object of class Seurat 
21157 features across 8595 samples within 1 assay 
Active assay: RNA (21157 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 3 dimensional reductions calculated: pca, umap, harmony